# VitaNexus-RX HGNN post-training audit

This notebook does **not** train or alter HGNN weights. It uses 2025 data for calibration and threshold selection, freezes the configuration, and only then evaluates the untouched 2026 holdout.

Attach the `VitaNexus_HGNN_Audit_Data.zip` package as a Kaggle Dataset, enable Internet for the Git clone, and select a GPU accelerator so the final frozen holdout inference is faster. Calibration stages are saved under `/kaggle/working/hgnn_audit_work`; rerunning a failed cell resumes completed calibration methods.

In [ ]:
from pathlib import Path
import importlib.util, json, os, shutil, subprocess, sys, zipfile

BRANCH = 'codex/faers-ml-clinical-integration'
REPOSITORY = Path('/kaggle/working/VitaNexus-RX')
if not REPOSITORY.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/Sravanramaraju/VitaNexus-RX.git', str(REPOSITORY)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPOSITORY, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPOSITORY, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPOSITORY, check=True)

required = {'pyarrow':'pyarrow', 'sklearn':'scikit-learn', 'torch_geometric':'torch-geometric', 'joblib':'joblib', 'scipy':'scipy'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)
sys.path.insert(0, str(REPOSITORY / 'ml' / 'src'))
print('Repository:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY, text=True).strip())

In [ ]:
archives = list(Path('/kaggle/input').rglob('VitaNexus_HGNN_Audit_Data.zip'))
extract_root = Path('/kaggle/working/hgnn_audit_input')
if archives and not extract_root.exists():
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(extract_root)

search_roots = [extract_root, Path('/kaggle/input')]
snapshots = [path for root in search_roots if root.exists() for path in root.rglob('VitaNexus-RX-ML') if (path / 'data/processed/faers/cohort.parquet').exists()]
caches = [path.parent for root in search_roots if root.exists() for path in root.rglob('selection_validation/metadata.json')]
assert snapshots, 'Attach the VitaNexus HGNN audit data package: snapshot folder was not found.'
assert caches, 'Attach the VitaNexus HGNN audit data package: prediction caches were not found.'
SNAPSHOT_ROOT = snapshots[0]
CACHE_ROOT = caches[0]
WORK_ROOT = Path('/kaggle/working/hgnn_audit_work')
REPORT_ROOT = Path('/kaggle/working/hgnn_audit_reports')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
print({'snapshot': str(SNAPSHOT_ROOT), 'cache': str(CACHE_ROOT), 'work': str(WORK_ROOT)})

In [ ]:
from vitanexus_ml.models.hgnn_post_training import create_checkpoint_integrity_manifest
from vitanexus_ml.models.hgnn_post_training_audit import run_preholdout_audit

INTEGRITY = WORK_ROOT / 'checkpoint_integrity.json'
create_checkpoint_integrity_manifest(SNAPSHOT_ROOT, INTEGRITY)
FROZEN = REPORT_ROOT / 'hgnn_post_training_frozen_manifest.json'
if FROZEN.exists():
    print('Pre-2026 configuration already frozen; skipping optimization.')
    preholdout = json.loads((REPORT_ROOT / 'hgnn_post_training_preholdout.json').read_text())
else:
    preholdout = run_preholdout_audit(INTEGRITY, CACHE_ROOT, WORK_ROOT, REPORT_ROOT)
print(json.dumps(preholdout, indent=2, default=str)[:12000])

In [ ]:
import torch
from vitanexus_ml.models.hgnn_post_training_audit import run_frozen_holdout_evaluation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
holdout = run_frozen_holdout_evaluation(INTEGRITY, FROZEN, CACHE_ROOT, WORK_ROOT, REPORT_ROOT, device_name=device)
print('Frozen 2026 evaluation complete on', device)
print(json.dumps({'metrics': holdout['metrics'], 'topK': holdout['topK']}, indent=2))

In [ ]:
delivery = Path('/kaggle/working/VitaNexus_HGNN_Audit_Results')
delivery.mkdir(exist_ok=True)
shutil.copytree(REPORT_ROOT, delivery / 'reports', dirs_exist_ok=True)
shutil.copytree(WORK_ROOT, delivery / 'resume_state', dirs_exist_ok=True)
archive = shutil.make_archive('/kaggle/working/VitaNexus_HGNN_Audit_Results', 'zip', delivery)
print('Download or save this Kaggle output:', archive)